# 10.1 多模态小模型基础与选型 (Multimodal Small Model Foundations)

## 📚 本章概览 (Overview)

**学习目标**：
- 理解多模态模型的典型架构（CNN / ViT / CLIP 双塔）
- 掌握轻量级视觉模型谱系及各自适用场景
- 理解视觉-语言对齐的核心方法（CLIP-style contrastive learning）
- 建立可量化的模型选型框架：精度 vs 延迟 vs 显存 vs 功耗

**核心问题**：几十种轻量模型到底怎么选？不同业务场景（安防/零售/医疗）的约束条件完全不同。

🏢 **业务场景**：你负责为公司的「通用多模态平台」做技术选型。该平台需要同时服务三个客户——安防客户要求低延迟实时检测（<100ms）、零售客户需要识别 200+ SKU、医疗客户要求高精度和可解释性。你的选型决策将直接影响平台的架构和成本。

**知识地图**：本章是整个 Module 10 的基础，后续的图像识别、视频理解、模型部署都依赖此处的选型决策。

**预计学习时间**：3-4 小时

## 🎯 动机与背景 (Motivation)

### 为什么需要专门的模型选型方法？

2024-2025 年，视觉模型的数量爆炸式增长——仅 HuggingFace 上就有 200+ 个图像分类模型。与此同时，边缘设备（Jetson、树莓派、手机）的算力虽然持续增长，但远跟不上模型规模的扩张。

这意味着：**选错模型的代价很高**。选了精度高但太慢的模型 → 边缘跑不动；选了够快但精度低的模型 → 业务指标不达标。

### 要解决的实际问题

1. 给定一个业务场景（延迟约束 + 精度需求 + 显存预算），如何在 10+ 个候选模型中快速收敛到 2-3 个
2. 如何用 benchmark 数据而非直觉来做决策
3. CLIP 等图文对齐模型在零样本场景下的可靠性如何评估

In [ ]:
# 🔬 Micro Practice 1: Load and benchmark lightweight models
# Goal: Build intuition for model size vs speed trade-offs

import timm
import torch
import time
import numpy as np

# TODO: Load MobileNetV3, EfficientNet-B0, TinyViT-5M
# TODO: Benchmark inference time on CPU and GPU
# TODO: Record parameter counts and FLOPs

print("Model benchmarking setup complete")

## 📖 理论基础 (Theory)

### 3.1 视觉编码器三大范式

**CNN (Convolutional Neural Network, 卷积神经网络)**：
- 局部感受野、权重共享、平移等变性
- 代表：MobileNetV3, EfficientNet
- 优势：推理效率高，量化友好；劣势：全局建模能力弱

**ViT (Vision Transformer, 视觉 Transformer)**：
- 图像 → Patch 序列 → Self-Attention
- 代表：TinyViT, FastViT, MobileViT
- 优势：全局感受野，多模态对齐自然；劣势：小模型时效率不如 CNN

**混合架构 (CNN + Transformer)**：
- 浅层 CNN 提取局部特征 + 深层 Transformer 全局建模
- 代表：ConvNeXt, MobileOne
- 优势：兼顾效率和精度

### 3.2 CLIP 图文对齐原理

CLIP (Contrastive Language-Image Pre-training, 对比语言-图像预训练) 使用双塔架构：
- 视觉编码器：图像 → embedding vector
- 文本编码器：文本 → embedding vector
- 对比损失 (InfoNCE)：拉近匹配图文对，推远非匹配对

这使得 CLIP 可以在零样本情况下完成分类——不需要在目标类别上训练，只需要用文本描述类别即可。

In [ ]:
# 🔬 Micro Practice 2: CLIP zero-shot classification
# Goal: Understand vision-language alignment through hands-on experiment

from transformers import CLIPProcessor, CLIPModel

# TODO: Load MobileCLIP or CLIP-ViT-B/32
# TODO: Run zero-shot classification on sample images
# TODO: Visualize similarity scores between image and text prompts

print("CLIP zero-shot classification setup")

In [ ]:
# 🔬 Micro Practice 3: Model benchmark matrix
# Goal: Build a quantitative model selection framework

# TODO: Create benchmark matrix:
# - Columns: Model, Params, FLOPs, CPU Latency, GPU Latency, Acc@1, Memory
# - Rows: MobileNetV3, EfficientNet-B0, TinyViT-5M, FastViT-T8, MobileCLIP-S0
# TODO: Plot accuracy vs latency Pareto frontier

print("Benchmark matrix setup")

In [ ]:
# 🔬 Micro Practice 4: Input resolution impact analysis
# Goal: Understand how resolution affects accuracy and speed

# TODO: Benchmark same model at 128, 160, 224, 288, 320 resolutions
# TODO: Plot resolution vs accuracy vs latency

print("Resolution analysis setup")

In [ ]:
# 🔬 Micro Practice 5: Video frame sampling strategies
# Goal: Compare fixed-interval vs keyframe vs motion-detection sampling

import cv2

# TODO: Implement 3 sampling strategies
# TODO: Compare frame count and information retention

print("Frame sampling comparison setup")

In [ ]:
# 🔬 Micro Practice 6: Multimodal embedding visualization
# Goal: Visualize image-text alignment in embedding space

from sklearn.manifold import TSNE
import matplotlib.pyplot as plt

# TODO: Extract image and text embeddings from CLIP
# TODO: Use t-SNE to visualize alignment

print("Embedding visualization setup")

In [ ]:
# 🔬 Micro Practice 7: Decision matrix exercise
# Goal: Apply model selection framework to a real scenario

# TODO: Given scenario constraints (latency < 100ms, accuracy > 80%, memory < 2GB)
# TODO: Score each candidate model and output recommendation with justification

print("Decision matrix exercise setup")

In [ ]:
# 🔬 Micro Practice 8: Business scenario matching
# Goal: Match models to security/retail/medical scenarios

# TODO: For each scenario, define constraints and select top-2 models
# TODO: Write a 1-page selection rationale

print("Scenario matching setup")

## 🔨 从零实现 (Implementation from Scratch)

### 视觉 Patch Embedding 的 NumPy 实现

理解 ViT 的第一步：一张图如何变成 token 序列。

In [ ]:
# NumPy implementation of Patch Embedding
import numpy as np

def patch_embedding_numpy(image, patch_size=16, embed_dim=768):
    """
    Convert image to patch embeddings using pure NumPy.
    
    Args:
        image: (H, W, C) numpy array
        patch_size: size of each patch
        embed_dim: output embedding dimension
    
    Returns:
        embeddings: (num_patches, embed_dim)
    """
    # TODO: Implement patch extraction and linear projection
    pass

# Test with a sample image
print("Patch embedding NumPy implementation")

In [ ]:
# NumPy implementation of Self-Attention for vision
def self_attention_numpy(x, W_q, W_k, W_v):
    """
    Self-attention for patch sequences.
    
    Args:
        x: (num_patches, embed_dim)
        W_q, W_k, W_v: projection matrices
    
    Returns:
        output: (num_patches, embed_dim)
        attention_weights: (num_patches, num_patches)
    """
    # TODO: Implement QKV projection and attention computation
    pass

print("Self-attention NumPy implementation")

## ⚙️ 工程化实现 (Engineering Implementation)

### PyTorch + timm 模型加载与评估

从 NumPy 理解原理后，使用工业级工具链进行工程化实践。

In [ ]:
# Production-grade model benchmarking with PyTorch + timm
import torch
import timm
import json
from time import perf_counter
from dataclasses import dataclass

@dataclass
class ModelBenchmark:
    """Benchmark result for a single model."""
    name: str
    params_m: float  # millions
    flops_g: float   # billions
    cpu_latency_ms: float
    gpu_latency_ms: float
    memory_mb: float
    acc1_imagenet: float

# TODO: Implement systematic benchmarking pipeline
# TODO: Generate comparison report

print("Engineering benchmarking pipeline setup")

In [ ]:
# Benchmark runner that outputs structured results
def benchmark_model(model_name, input_size=224, num_runs=100):
    """
    Comprehensive model benchmark.
    
    Args:
        model_name: timm model name
        input_size: input image size
        num_runs: number of inference runs for averaging
    
    Returns:
        ModelBenchmark: structured benchmark result
    """
    # TODO: Implement
    pass

# Run benchmarks on candidate models
candidates = ['mobilenetv3_small_100', 'efficientnet_b0', 'tiny_vit_5m_224']
print("Benchmark runner ready")

## 🚀 综合项目 (Capstone Project)

### 项目：多模态平台技术选型报告

**需求**：为「通用多模态平台」完成完整的技术选型，覆盖 3 个下游场景。

**基础实现（必做）**：
1. 建立候选模型池（≥ 8 个模型）
2. 定义基准测试协议（数据集、指标、环境）
3. 完成全量 benchmark（精度、CPU 延迟、GPU 延迟、显存）
4. 输出每个场景（安防/零售/医疗）的推荐模型和理由

**进阶挑战（选做）**：
1. 实现自动选型决策引擎（输入约束 → 输出推荐模型排名）
2. 绘制三维 Pareto 前沿图（精度 × 延迟 × 显存）
3. 加入功耗维度，对比 Jetson 和 x86 平台的差异化选型

In [ ]:
# 🚀 Capstone: Model Selection Report Generator
# TODO: Implement complete benchmarking and recommendation system

print("Capstone project setup")

## ❓ 常见问题与调试 (FAQ & Debugging)

### Q1: timm 模型加载后 forward 报 shape mismatch？
检查输入尺寸是否匹配模型默认配置。使用 `model.default_cfg` 查看期望的 input_size。

### Q2: CLIP 零样本分类效果波动很大？
Prompt 工程至关重要。"a photo of {class}" vs "a blurry photo of {class}" 可能差 5 个点。建议做 prompt ensemble。

### Q3: 模型参数量小但推理为什么慢？
参数量 ≠ 推理速度。FLOPs、内存带宽、算子优化程度都有影响。例如深度可分离卷积参数量少但内存访问模式不友好。

### Q4: CPU 推理如何加速？
- ONNX Runtime 比原生 PyTorch 快 2-5x
- OpenVINO (Intel CPU) 额外加速 30-50%
- 关注线程数设置（通常设为物理核心数）

### Q5: 移动端和边缘端有什么区别？
- 移动端：手机/NPU，Core ML / NNAPI，功耗敏感
- 边缘端：Jetson/工控机，GPU/TPU，散热条件更好
- 选型时区分目标平台，不能混用 benchmark

## 📝 总结与展望 (Summary)

### 核心要点回顾
1. 视觉模型三大范式（CNN / ViT / Hybrid）各有适用场景，没有银弹
2. CLIP 双塔架构提供了零样本能力，但精度-效率 trade-off 需要量化评估
3. 模型选型是工程决策而非学术实验——用 benchmark 数据说话
4. 同一个模型在不同硬件上的表现可能天差地别，必须实测

### 与后续章节的联系
- **10.2 图像识别**：基于本章选定的模型进行实战训练
- **10.3 视频理解**：在选型基础上加入时间维度
- **10.4 边缘部署**：将选定模型量化并部署到边缘

### 💡 思考题
1. 如果安防客户同时需要检测和识别两个任务，是选两个专用模型还是一个多任务模型？决策依据是什么？
2. CLIP 的零样本能力在实际业务中可靠吗？什么时候应该放弃零样本转而做微调？
3. 模型选型后 6 个月，出现了更好的新模型。什么时候值得切换？切换的成本（重新标注、重新训练、重新部署）如何量化？

### 下一步
进入 10.2 图像识别实践，将选定的模型用于真实场景训练。